In [0]:
# ============================================================
# PFIN | Phase 1 | Bronze Ingestion
# Notebook:  01_ingest_bronze
# Source:    landing container (ADLS Gen2)
# Target:    pfin_dev.bronze.elections_canada_contributions_raw
# ============================================================

import re
from pyspark.sql import functions as F
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────────
STORAGE_ACCOUNT = "pfincanadacentralsa"
LANDING_BASE    = f"abfss://landing@{STORAGE_ACCOUNT}.dfs.core.windows.net"
TARGET_CATALOG  = "pfin_dev"
TARGET_SCHEMA   = "bronze"
TARGET_TABLE    = "elections_canada_contributions_raw"
FULL_TABLE_NAME = f"{TARGET_CATALOG}.{TARGET_SCHEMA}.{TARGET_TABLE}"

SOURCE_FILES = [
    f"{LANDING_BASE}/od_cntrbtn_de_e_2025.csv",
    f"{LANDING_BASE}/od_cntrbtn_de_e_2026.csv",
]

# ── READ ─────────────────────────────────────────────────────
df_raw = (
    spark.read
    .option("header", "true")
    .option("encoding", "iso-8859-1")
    .option("inferSchema", "false")
    .option("multiLine", "false")
    .csv(SOURCE_FILES)
)

# ── CLEAN COLUMN NAMES ───────────────────────────────────────
def clean_col_name(name):
    name = name.encode("iso-8859-1").decode("utf-8", errors="ignore")
    name = re.sub(r'[^\x00-\x7F]+', '', name)
    name = name.strip()
    name = re.sub(r'[ ,;{}()\n\t=\/]+', '_', name)
    name = name.replace('-', '_')
    name = name.strip('_').lower()
    return name

cleaned_cols = [clean_col_name(c) for c in df_raw.columns]
print("Cleaned columns:", cleaned_cols)

# ── ADD METADATA COLUMNS ─────────────────────────────────────
df_bronze = (
    df_raw.toDF(*cleaned_cols)
    .withColumn("_source_file", F.col("_metadata.file_path"))
    .withColumn("_ingested_at", F.current_timestamp())
)

# ── WRITE TO DELTA ───────────────────────────────────────────
(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(FULL_TABLE_NAME)
)

# ── VALIDATE ─────────────────────────────────────────────────
row_count = spark.table(FULL_TABLE_NAME).count()
col_count = len(spark.table(FULL_TABLE_NAME).columns)

print(f"✅ Table : {FULL_TABLE_NAME}")
print(f"   Rows  : {row_count:,}")
print(f"   Cols  : {col_count}")
print(f"   Done  : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Cleaned columns: ['political_entity', 'recipient_id', 'recipient', 'recipient_last_name', 'recipient_first_name', 'recipient_middle_initial', 'political_party_of_recipient', 'electoral_district', 'electoral_event', 'fiscal_election_date', 'form_id', 'financial_report', 'part_number_of_return', 'financial_report_part', 'contributor_type', 'contributor_name', 'contributor_last_name', 'contributor_first_name', 'contributor_middle_initial', 'contributor_city', 'contributor_province', 'contributor_postal_code', 'contribution_received_date', 'monetary_amount', 'non_monetary_amount', 'contribution_given_through', 'leadership_contestant']
✅ Table : pfin_dev.bronze.elections_canada_contributions_raw
   Rows  : 284,136
   Cols  : 29
   Done  : 2026-06-01 01:50:35
